# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/james-kalaw/FlyRank-ML-Intern-Kalaw/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: signal analysis.** It is not one of the four types above, and that is on purpose.

The framing skill lists four task types. My question ("which common SEO beliefs hold up in this data?") matches the fourth one, *"Which signals travel together?" → signal analysis*: no trained model, measured with effect sizes and grouped comparisons.

It is *supervised in a loose sense*: every signal is compared against the same observed outcome (whether a page is declining). But I am not building a classifier that predicts decline for new pages, and I am not ranking pages. The output is a **verdict per signal**, so it is closest to *scoring signals*, not scoring pages.

Why not the others:
- **Classification / ranking:** those answer "which pages will decline / which first?" (Lane 2). My customer needs "which beliefs are true?", which a page-level score doesn't answer.
- **Clustering:** finds page types with no outcome at all (Lane 3). I need an outcome to judge each belief against.

The code below loads the data and builds the outcome column used in every section.

In [1]:
# Setup: load the starter CSV (clones my repo first if the data folder is missing)
import os, subprocess
import numpy as np
import pandas as pd

DATA = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(DATA):
    if not os.path.isdir("repo"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/james-kalaw/FlyRank-ML-Intern-Kalaw.git", "repo"],
                       check=True)
    os.chdir("repo")

df = pd.read_csv(DATA)

# The outcome every signal is compared against
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
BASE_RATE = df["is_declining"].mean() * 100

print(f"{len(df):,} pages, {df['client_id'].nunique()} clients")
print(f"Base rate: {BASE_RATE:.1f}% of pages are declining")

30,000 pages, 32 clients
Base rate: 54.2% of pages are declining


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Outcome (proxy): `is_declining` = 1 when `trend_direction == "down"`, else 0.** Base rate: about 54% of pages.

**Observed or defined?** Both, in two layers, and I want to be honest about it:
1. **Observed:** `trend_pct`, the measured % change in Google impressions between the last 30 days and the 30 days before (`(last30 − prev30) / prev30 × 100`).
2. **Defined by a rule:** `trend_direction` cuts that number into buckets. Below −20% is `down`.

So the label is a **rule applied to an observed measurement**, not a rule someone invented about what *should* decline. That's acceptable for a signal report, but it means the −20% cut-off is a choice. A page at −19% counts as "not declining." The code below checks the cut-off and that `trend_pct` really comes from the impression windows.

**Two limits I'm accepting for now:**
- **Same time window.** In the starter CSV, the signals (e.g. `impressions_90d`, `ctr`) and the outcome come from the *same* 90 days. My results will show which signals **go together** with decline, not which ones **come before** it. The Week 3 warehouse has daily data, so I can measure signals in an earlier window and decline in a later one.
- **Columns I will never use as signals**, because they are the outcome or its ingredients: `trend_direction`, `trend_pct`, and the six `*_last_30d` / `*_prev_30d` columns. `impressions_90d` also overlaps the outcome window, so any result on it gets a warning label.

In [2]:
# Where the label comes from

# 1. The cut-off: trend_pct range inside each trend_direction bucket
print("trend_pct (%) by trend_direction:")
print(df.groupby("trend_direction")["trend_pct"].agg(["count", "min", "max"]).round(1).to_string())

# 2. trend_pct really is the change between the two 30-day impression windows
prev = df["impressions_prev_30d"]
last = df["impressions_last_30d"]
has_prev = prev > 0
recomputed = (last - prev) / prev * 100
match = ((recomputed - df["trend_pct"]).abs() <= 0.1)[has_prev].mean() * 100
print(f"\ntrend_pct matches (last30 - prev30) / prev30 on {match:.1f}% of pages with prev30 > 0")
print(f"Pages with prev30 = 0 (trend_pct blank, 'new' or 'flat'): {(~has_prev).sum():,}")

# 3. Columns that are the outcome or its ingredients: never used as signals
LABEL_COLS = ["trend_direction", "trend_pct", "is_declining"]
LABEL_INGREDIENTS = [c for c in df.columns if c.endswith("_last_30d") or c.endswith("_prev_30d")]
print("\nNever used as signals:", LABEL_COLS + LABEL_INGREDIENTS)

trend_pct (%) by trend_direction:
                 count    min      max
trend_direction                       
down             16262 -100.0    -20.0
flat                 0    NaN      NaN
new                  0    NaN      NaN
stable            5962  -20.0     20.0
up                4388   20.0  44900.0

trend_pct matches (last30 - prev30) / prev30 on 100.0% of pages with prev30 > 0
Pages with prev30 = 0 (trend_pct blank, 'new' or 'flat'): 3,388

Never used as signals: ['trend_direction', 'trend_pct', 'is_declining', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: the decline-rate gap.** For one signal at a time:

> **gap = (decline rate among the top 25% of pages on that signal) − (decline rate among the bottom 25%)**, in percentage points.

Example: if the longest 25% of pages decline at 50% and the shortest 25% at 58%, the gap is **−8 points**. Longer pages decline *less*.

**What counts as "good" (decided now, before looking at most signals):**

| Gap size | Verdict |
|---|---|
| 5 points or more (either sign) | **notable**: worth a closer look |
| 2 to under 5 points | **weak** |
| under 2 points | **none**: the belief doesn't show up |

These are in absolute percentage points against a ~54% base rate. A notable signal also needs at least ~500 pages in each group, and later it must keep its direction when I check it client by client (Week 6), so it isn't driven by one or two clients.

**Why this metric:** it's in plain English ("pages with lots of X decline 8 points less"), it's a grouped comparison (which caught the CTR cliff that a correlation missed), and it shows the base rate right next to it.

**Known weaknesses:** comparing only the two ends ignores the middle 50%, so a U-shape (both ends high) can show a gap near zero. I'll catch that by also looking at tier tables (Section 5). Columns with many zeros (like `search_volume`) can give unequal group sizes, so the code prints the group sizes.

The code runs the metric on four signals I already looked at in Week 1, as a first baseline run.

**Result (fill in after Run all):** Three signals were **notable**: content age (gap **−16.4**), search volume (**−12.2**), and word count (**+10.3**). In this sample, older pages and pages on higher-volume topics declined less often, while longer pages declined more often. Avg position was only **weak** (**−3.7**), but Section 5 shows that comparing just the two ends hides a hill-shaped pattern. The word-count result differs from my Week 1 median comparison (which looked flat), so I will re-check it by `content_type`, since 25.7% of pages have no word count.

In [3]:
# The metric: decline-rate gap between the top and bottom 25% of a signal

def decline_gap(data, signal, outcome="is_declining"):
    d = data[[signal, outcome]].dropna()
    low_cut, high_cut = d[signal].quantile([0.25, 0.75])
    bottom = d.loc[d[signal] <= low_cut, outcome]
    top = d.loc[d[signal] >= high_cut, outcome]
    gap = (top.mean() - bottom.mean()) * 100
    verdict = "notable" if abs(gap) >= 5 else ("weak" if abs(gap) >= 2 else "none")
    return {"signal": signal,
            "pages_bottom": len(bottom), "pages_top": len(top),
            "bottom_decline_%": round(bottom.mean() * 100, 1),
            "top_decline_%": round(top.mean() * 100, 1),
            "gap_pts": round(gap, 1), "verdict": verdict}

# avg_position = 0 means "no data", not rank zero, so drop those pages for that signal
with_position = df[df["avg_position"] > 0]

rows = [decline_gap(df, "content_age_days"),
        decline_gap(df, "word_count"),
        decline_gap(df, "search_volume"),
        decline_gap(with_position, "avg_position")]
print(f"Base rate: {BASE_RATE:.1f}% declining\n")
print(pd.DataFrame(rows).set_index("signal").to_string())

Base rate: 54.2% declining

                  pages_bottom  pages_top  bottom_decline_%  top_decline_%  gap_pts  verdict
signal                                                                                      
content_age_days          7518       7764              60.0           43.7    -16.4  notable
word_count                5576       5578              49.7           60.0     10.3  notable
search_volume            11081       9140              63.2           51.1    -12.2  notable
avg_position              7412       7227              55.1           51.3     -3.7     weak


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one page** (one content item), with its metrics over a trailing 90-day window. Pages belong to clients (websites), and a client has many pages.

The code checks this instead of assuming it:
- every `content_id` appears exactly once, so one row really is one page;
- how many pages each client has, because a few big clients could dominate any average;
- the **role of every column**: ID (grouping only), outcome, outcome ingredient (excluded), metadata (excluded per the data dictionary), or **signal** (allowed to be tested). It also shows how much of each is missing.

I'm showing the table's structure, not raw rows, because the rows aren't needed to see the shape. That also keeps row-level data out of this notebook's outputs.

**Result (fill in after Run all):** One row is one page (**30,000 rows, 30,000 unique pages**). There are **32 signals** I can test; 13 columns are excluded (IDs, outcome, outcome ingredients, metadata). The most missing signals are `word_count` / `char_count` and their tiers (**25.7%**), then the keyword columns (~8%). Clients are very uneven: from **3** to **7,008** pages each, and the **top 3 clients hold 43.3%** of all pages, so every finding needs a client-by-client check later (Week 6).

In [4]:
# Unit of analysis: one row = one page

print(f"Rows: {len(df):,} | unique content_id: {df['content_id'].nunique():,}")
print("One row per page:", df["content_id"].is_unique)

pages_per_client = df.groupby("client_id").size()
print(f"\nPages per client: min {pages_per_client.min():,}, "
      f"median {int(pages_per_client.median()):,}, max {pages_per_client.max():,}")
print(f"Biggest 3 clients hold {pages_per_client.nlargest(3).sum() / len(df):.1%} of all pages")

# Role of every column
ID_COLS = ["content_id", "client_id"]
METADATA_EXCLUDED = ["provider_used", "model_used"]  # data dictionary: "Not a model feature"
role = {}
for c in df.columns:
    if c in ID_COLS: role[c] = "id (grouping only)"
    elif c in LABEL_COLS: role[c] = "outcome"
    elif c in LABEL_INGREDIENTS: role[c] = "outcome ingredient (excluded)"
    elif c in METADATA_EXCLUDED: role[c] = "metadata (excluded)"
    else: role[c] = "signal"

columns = pd.DataFrame({"role": pd.Series(role),
                        "dtype": df.dtypes.astype(str),
                        "missing_%": (df.isna().mean() * 100).round(1)})
columns = columns.sort_values(["role", "missing_%"], ascending=[True, False])
print("\nColumns by role:")
print(columns["role"].value_counts().to_string())
print()
print(columns.to_string())

SIGNALS = columns.index[columns["role"] == "signal"].tolist()

Rows: 30,000 | unique content_id: 30,000
One row per page: True

Pages per client: min 3, median 567, max 7,008
Biggest 3 clients hold 43.3% of all pages

Columns by role:
role
signal                           32
outcome ingredient (excluded)     6
outcome                           3
id (grouping only)                2
metadata (excluded)               2

                                                 role    dtype  missing_%
content_id                         id (grouping only)   object        0.0
client_id                          id (grouping only)   object        0.0
provider_used                     metadata (excluded)   object       71.5
model_used                        metadata (excluded)   object       19.1
trend_pct                                     outcome  float64       11.3
trend_direction                               outcome   object        0.0
is_declining                                  outcome    int64        0.0
impressions_last_30d    outcome ingredient (exclud

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Honest answer: in my lane, I'm not claiming ML beats a rule.** Lane 1 doesn't train a model. The framing skill says ML only earns its place when the pattern is real but too messy to write by hand. What I can show is that **the folklore rules teams already use** (simple if-statements like "old pages decline") **are too simple for this data**. That is why a careful signal-by-signal check is needed instead of a rule of thumb.

Evidence I already have:
- **Backwards patterns:** declining pages have the *youngest* median age (216 days vs 300 for stable). The rule "old content decays" points the wrong way.
- **Non-linear patterns:** position vs CTR has almost no linear correlation (−0.073), yet CTR drops about 6× from page 1 to deep results. An if-statement with one cut-off, or a straight-line check, misses that shape.
- **Interacting signals:** in Week 2 the decision tree split on `impressions_90d` first and *then* on `content_age_days`. Age only mattered once a page had some traffic, so one signal's effect depends on another.

The code tests this directly:
1. scores a folklore rule ("pages 365+ days old are the ones declining") against the base rate;
2. shows decline rate across age tiers and position tiers, to see whether it moves in a straight line or bends;
3. crosses age tier with impression tier, to see whether the age pattern changes depending on traffic level (an interaction).

**Result (fill in after Run all):** The rule "pages 365+ days old are declining" flagged 6,360 pages, and only **42.6%** of them were declining, **below the 54.2% base rate**: the rule points the wrong way. By age, decline falls in a nearly **straight line** (66.9% → 62.6% → 51.5% → 42.6% from newest to oldest). By position it is **hill-shaped**: top 3 (**24.1%**) and deep (**34.4%**) decline least, the middle tiers most (up to **61.0%** for striking). Age and traffic **interact**: for low-traffic pages, newest vs oldest is **65.6% vs 40.1%** (about 25 points), but for excellent-traffic pages it is only **46.2% vs 42.1%** (about 4 points). These are observed associations in one 90-day window, not causes.

In [5]:
# Why a one-line rule isn't enough: backwards, bent, and interacting patterns

def rate_by(col, order=None):
    t = df.groupby(col)["is_declining"].agg(pages="size", decline_rate="mean")
    t["decline_rate"] = (t["decline_rate"] * 100).round(1)
    if order is not None:
        t = t.reindex([o for o in order if o in t.index])
    return t

# 1. A folklore rule, scored against the base rate
old = df["content_age_days"] >= 365
print(f"Rule 'pages 365+ days old are declining': {old.sum():,} pages flagged, "
      f"{df.loc[old, 'is_declining'].mean() * 100:.1f}% of them declining "
      f"(base rate {BASE_RATE:.1f}%)")

# 2. Does decline move in a straight line across tiers?
age_order = (df[["age_tier", "age_tier_order"]].drop_duplicates()
             .sort_values("age_tier_order")["age_tier"].tolist())
pos_order = ["top_3", "page_1", "striking", "page_3_5", "deep", "no_data"]
print("\nDecline rate (%) by age tier:")
print(rate_by("age_tier", age_order).to_string())
print("\nDecline rate (%) by position tier:")
print(rate_by("position_tier", pos_order).to_string())

# 3. Interaction: does the age pattern change with traffic level?
imp_order = ["no_data", "none", "low", "moderate", "good", "excellent"]
rates = df.pivot_table(index="age_tier", columns="impression_tier",
                       values="is_declining", aggfunc="mean") * 100
sizes = df.pivot_table(index="age_tier", columns="impression_tier",
                       values="is_declining", aggfunc="size")
rates = rates.where(sizes >= 100)  # hide cells with fewer than 100 pages (too noisy)
rates = rates.reindex(index=age_order,
                      columns=[c for c in imp_order if c in rates.columns]).round(1)
print("\nDecline rate (%) by age tier x impression tier (blank = under 100 pages):")
print(rates.to_string())

Rule 'pages 365+ days old are declining': 6,360 pages flagged, 42.6% of them declining (base rate 54.2%)

Decline rate (%) by age tier:
          pages  decline_rate
age_tier                     
31-90       492          66.9
91-180    11780          62.6
181-365   11368          51.5
365+       6360          42.6

Decline rate (%) by position tier:
               pages  decline_rate
position_tier                     
top_3           2321          24.1
page_1         11814          57.0
striking        7304          61.0
page_3_5        7242          56.2
deep            1319          34.4

Decline rate (%) by age tier x impression tier (blank = under 100 pages):
impression_tier   low  moderate  good  excellent
age_tier                                        
31-90            65.6      66.4  71.4        NaN
91-180           52.9      71.6  66.5       46.2
181-365          39.6      63.7  57.6       48.3
365+             40.1      44.0  43.4       42.1


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.